<a href="https://colab.research.google.com/github/hector-carpi/-everpeak-analysis/blob/main/Dataset_EverPeak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
*dataset limpio de EverPeak y analizaremos las columnas numéricas relevantes.

**Análisis estadístico
# importar librerías y dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


df = pd.read_csv("/datasets/everpeak_clean.csv")

Comenzamos con una revisión general de todas las variables.
Por ahora nos enfocaremos en las columnas numéricas.
Identificamos las columnas relevantes como: price, quantity, order_value y customer_age.
E incluimos una breve descripción de ellas y por qué las analizamos.

**1. Visión general
df.info()
df.head()
Las columnas price, quantity, order_value y customer_age son de tipo INT.
Las analizamos porque permiten identificar patrones, distribuciones y valores atípicos útiles para el análisis.

Posteriormente, calculamos sus estadísticas descriptivas básicas:
Observamos la cantidad de valores, la media, el mínimo y máximo, los cuartiles y la mediana.

**2. Estadísticas descriptivas
num_cols = ["price", "quantity", "order_value", "customer_age"]
df[num_cols].describe()

Después pasamos a la visualización diagnóstica
Aquí, evaluamos la distribución de cada variable mediante un histograma y un boxplot, para ver su forma y detectar la presencia de sesgo.
En price, quantity y order_value se detectan valores atípicos hacia la derecha, mientras que en customer_age no se observan.

**3. Visualización diagnóstica
# Graficar histogramas
for col in num_cols:
    plt.figure(figsize=(9, 3))
    sns.histplot(df[col], bins=60)
    plt.title(f'Distribución de la variable {col}')
    plt.show()
# Graficar boxplots
for col in num_cols:
    plt.figure(figsize=(9,3))
    sns.boxplot(x=df[col])
    plt.title(f'Distribución de la variable {col}')
    plt.show()

A continuación, pasamos a la identificación formal de outliers.
Además de identificarlos con el boxplot, debemos usar el IQR o z-score para realizar un doble chequeo y analizar los valores atípicos en caso de que existan.

**4. Identificación formal de outliers
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    print(col, "IQR: ", IQR)

    upper = Q3 + 1.5*IQR
    lower = Q1 - 1.5*IQR
    display(df[(df['price'] > upper) | (df['price'] < lower)])

Si hay valores atípicos presentes, seguimos con la clasificación de outliers.
Para cada tipo de valor atípico debemos indicar si:
Se trata de un error y debe ser eliminado.
Si es un valor posible a mantener.
O, si debemos caparlo.
Recuerda indicar y justificar en cada caso qué haremos y por qué:
En este ejemplo, tenemos únicamente valores atípicos altos que son posibles pero poco comunes; para que el análisis represente al cliente típico, capamos los valores mayores al percentil del 99.

**5. Clasificación y tratamiento de outliers: Drop, Keep o Cap
Drop: cuando el valor es imposible
Keep: es un valor posible a mantener
Winsorization: cuando es extremo pero posible
**5.1 Winsorization (valores extremos pero posibles)
Para conservar la información del cliente típico, capamos los valores que estén por encima del percentil del 99.
for col in num_cols:
    p99 = df[col].quantile(0.99)
    df[f'{col}_capped'] = np.clip(df[col], None, p99)


df[["price","price_capped", "quantity","quantity_capped",
    "order_value","order_value_capped", "customer_age","customer_age_capped"]].head()

Antes de terminar, es importante resaltar las estadísticas post-tratamiento.
Esto nos permite comparar el antes y el después, destacando ventajas, inferencias y posibles puntos de conflicto.

**6. Estadísticas post-tratamiento
df[["price","price_capped", "quantity","quantity_capped",
    "order_value","order_value_capped", "customer_age","customer_age_capped"]].describe()

Finalmente, cerramos con una conclusión ejecutiva para cada característica, es decir una frase clara y comprensible para el área de negocio, tal como se muestra en el ejemplo en pantalla.

**7. Conclusión ejecutiva
**7.1 Columna price:
El 50% de los precios se encuentra entre 218 y 847.
Los valores >847 USD corresponden al 25% de los precios más altos.
Se realizó winsorización al percentil 99 para estabilizar métricas sin perder estos segmentos.

Con esta estructura, tu análisis será claro, completo y fácil de interpretar.
